# 🛡️ Aegis Markets — BiLSTM Training Pipeline (v2, Stationary)

**What's different in this version:** all features are converted to *stationary* form — returns, ratios, and normalized oscillators instead of raw price levels. This generally helps an LSTM generalize better, since the model sees relative movement patterns rather than absolute price magnitudes (which drift over a 7-8 year window).

**Instructions:**
1. Open [Google Colab](https://colab.research.google.com), create a new notebook.
2. Runtime → Change runtime type → **T4 GPU**.
3. Run cells top to bottom. Change `TICKER` in Step 1 and re-run to train the other asset.

## Step 1: Install libraries

In [ ]:
import os
os.system("pip install -q yfinance scikit-learn pandas numpy matplotlib tensorflow")
print("✅ Libraries installed.")

## Step 2: Imports

In [ ]:
import pickle
import json
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout, BatchNormalization, Layer
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Step 3: Configuration

Change `TICKER` here to select your asset:
- `^NSEI` → Nifty 50
- `BZ=F` → Brent Crude Oil

Run the notebook twice — once per asset — to get both models.

In [ ]:
# ═══════════════════════════════════════════
#   CHANGE THIS TO SWITCH ASSET
TICKER = "^NSEI"   # "^NSEI" or "BZ=F"
# ═══════════════════════════════════════════

YEARS_DATA   = 8
LOOKBACK     = 30
EPOCHS       = 120
BATCH_SIZE   = 32
TEST_SPLIT   = 0.15

CLEAN_TICKER = TICKER.replace('^', '').replace('=', '')
ASSET_NAME   = 'Nifty 50' if TICKER == '^NSEI' else 'Brent Crude Oil'
MODEL_FILE   = f'lstm_model_{CLEAN_TICKER}.h5'
SCALER_FILE  = f'scaler_{CLEAN_TICKER}.pkl'
TARGET_SCALER_FILE = f'target_scaler_{CLEAN_TICKER}.pkl'
CONFIG_FILE  = f'feature_config_{CLEAN_TICKER}.json'

START_DATE = (datetime.now() - timedelta(days=YEARS_DATA * 365)).strftime('%Y-%m-%d')
END_DATE   = datetime.now().strftime('%Y-%m-%d')

print(f'📈 Training Asset  : {ASSET_NAME} ({TICKER})')
print(f'📅 Historical Range: {START_DATE} to {END_DATE}')
print(f'👁️ Window Lookback  : {LOOKBACK} days')

## Step 4: Download OHLCV + macro cross-asset data

- Nifty: uses Brent Crude, USD/INR, India VIX, Gold, S&P 500
- Crude: uses USD Index (DXY), Gold, S&P 500, Natural Gas, VIX

In [ ]:
def safe_download(ticker, start, end, name):
    try:
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if df.empty:
            print(f"  ⚠️  {name}: Empty data")
            return None
        print(f"  ✅ {name:15s}: {len(df)} days fetched")
        return df
    except Exception as e:
        print(f"  ❌ {name} failed: {e}")
        return None

print("📥 Fetching asset price records...")
stock_raw = safe_download(TICKER, START_DATE, END_DATE, ASSET_NAME)
if stock_raw is None:
    raise ValueError(f"Failed to download data for {TICKER}")

print("\n📥 Fetching macroeconomic indices...")
if TICKER == '^NSEI':
    macro_tickers = {
        'USDINR=X': 'USD_INR',
        '^INDIAVIX': 'India_VIX',
        'GC=F':     'Gold',
        'BZ=F':     'Crude_Oil',
        '^GSPC':    'SP500',
    }
else:
    macro_tickers = {
        'DX-Y.NYB': 'DXY',
        'GC=F':     'Gold',
        '^GSPC':    'SP500',
        'NG=F':     'NatGas',
        '^VIX':     'VIX',
    }

macro_series_dict = {}
for t, name in macro_tickers.items():
    m_df = safe_download(t, START_DATE, END_DATE, name)
    if m_df is not None:
        close_series = m_df['Close']
        if isinstance(close_series, pd.DataFrame):
            close_series = close_series.iloc[:, 0]
        macro_series_dict[name] = close_series

print(f"\n✅ Total macro signals loaded: {len(macro_series_dict)}")

## Step 5: Stationary feature engineering

Every feature here is a **return, ratio, or normalized oscillator** — not a raw price level. That's the core difference from v1: `Close_to_SMA10` (a ratio around 1.0) generalizes across price regimes far better than raw `SMA_10` (which was in the thousands and drifted over 8 years).

In [ ]:
print("🛠️ Processing stationary feature transformations...")
df = stock_raw.copy()

# 1. Price Returns (stationary relative shifts)
df['Ret_Close']  = df['Close'].pct_change(1)
df['Ret_Open']   = df['Open'].pct_change(1)
df['Ret_High']   = df['High'].pct_change(1)
df['Ret_Low']    = df['Low'].pct_change(1)
df['Ret_Volume'] = df['Volume'].pct_change(1).replace([np.inf, -np.inf], 0.0).fillna(0.0)

# 2. Moving average RATIOS (normalized by price → stationary)
df['SMA_10'] = df['Close'].rolling(10).mean()
df['SMA_20'] = df['Close'].rolling(20).mean()
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()

df['Close_to_SMA10'] = df['Close'] / (df['SMA_10'] + 1e-9)
df['Close_to_SMA20'] = df['Close'] / (df['SMA_20'] + 1e-9)
df['Close_to_EMA20'] = df['Close'] / (df['EMA_20'] + 1e-9)
df['Close_to_EMA50'] = df['Close'] / (df['EMA_50'] + 1e-9)

# 3. Normalized oscillators
delta = df['Close'].diff()
gain  = delta.where(delta > 0, 0).rolling(14).mean()
loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain / (loss + 1e-9)))
df['RSI_Scaled'] = df['RSI'] / 100.0

ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema12 - ema26
df['MACD_Price_Ratio'] = df['MACD'] / (df['Close'] + 1e-9)
df['Momentum_5'] = df['Close'].pct_change(5)

# 4. Volatility & volume shifts
std20 = df['Close'].rolling(20).std()
df['BB_Width'] = (4 * std20) / (df['SMA_20'] + 1e-9)
df['Rolling_Vol'] = df['Close'].pct_change().rolling(10).std()

obv = [0]
for i in range(1, len(df)):
    if df['Close'].iloc[i] > df['Close'].iloc[i-1]:
        obv.append(obv[-1] + df['Volume'].iloc[i])
    elif df['Close'].iloc[i] < df['Close'].iloc[i-1]:
        obv.append(obv[-1] - df['Volume'].iloc[i])
    else:
        obv.append(obv[-1])
df['OBV'] = np.array(obv)
df['OBV_pct'] = df['OBV'].pct_change(1).replace([np.inf, -np.inf], 0.0).fillna(0.0)

# 5. Macro returns
macro_feature_names = []
for name, series in macro_series_dict.items():
    col_name = f'Macro_{name}_5d'
    pct = series.pct_change(5).rename(col_name)
    df = df.merge(pct, left_index=True, right_index=True, how='left')
    df[col_name] = df[col_name].fillna(0.0)
    macro_feature_names.append(col_name)

df.dropna(inplace=True)

FEATURE_COLS = [
    'Ret_Close', 'Ret_Open', 'Ret_High', 'Ret_Low', 'Ret_Volume',
    'Close_to_SMA10', 'Close_to_SMA20', 'Close_to_EMA20', 'Close_to_EMA50',
    'RSI_Scaled', 'MACD_Price_Ratio', 'Momentum_5', 'BB_Width', 'Rolling_Vol', 'OBV_pct'
] + macro_feature_names

print(f"✅ Preprocessing done. Total stationary features: {len(FEATURE_COLS)}")
print(f"📌 Features: {FEATURE_COLS}")
df[FEATURE_COLS].tail(3)

## Step 6: Train/test split & scaling

In [ ]:
data_matrix = df[FEATURE_COLS].values.astype(np.float32)
n_features  = data_matrix.shape[1]

# 85/15 split FIRST — test set stays unseen
split = int(len(data_matrix) * (1 - TEST_SPLIT))
train_raw = data_matrix[:split]
test_raw  = data_matrix[split:]

# Scale to [-1, 1] since returns/ratios are centered around 0 / 1, not [0, max]
scaler = MinMaxScaler(feature_range=(-1, 1))
train_data = scaler.fit_transform(train_raw)
test_data  = scaler.transform(test_raw)

with open(SCALER_FILE, 'wb') as f:
    pickle.dump(scaler, f)

print(f'✅ Scaler fit on TRAIN ONLY, saved to: {SCALER_FILE}')
print(f'   Train records: {len(train_data)} | Test records: {len(test_data)}')

## Step 7: Build sequences (30-day lookback) & target scaling

In [ ]:
def create_sequences(scaled_data, raw_close, lookback):
    """X = lookback window of scaled features. y = next-day RETURN (not price) —
    this avoids the model learning to just echo yesterday's close."""
    X, y, prev_close = [], [], []
    for i in range(lookback, len(scaled_data)):
        X.append(scaled_data[i - lookback:i, :])
        ret = (raw_close[i] - raw_close[i - 1]) / (raw_close[i - 1] + 1e-9)
        y.append(ret)
        prev_close.append(raw_close[i - 1])
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.float32),
            np.array(prev_close, dtype=np.float32))

X_train, y_train_ret, prev_close_train = create_sequences(train_data, df['Close'].values[:split], LOOKBACK)
X_test,  y_test_ret,  prev_close_test  = create_sequences(test_data,  df['Close'].values[split:], LOOKBACK)

target_scaler = StandardScaler()
y_train = target_scaler.fit_transform(y_train_ret.reshape(-1, 1)).flatten().astype(np.float32)
y_test  = target_scaler.transform(y_test_ret.reshape(-1, 1)).flatten().astype(np.float32)

with open(TARGET_SCALER_FILE, 'wb') as f:
    pickle.dump(target_scaler, f)

print(f'✅ X_train: {X_train.shape}  (samples x timesteps x features)')
print(f'✅ X_test : {X_test.shape}')
print(f'✅ Target scaler saved to: {TARGET_SCALER_FILE}')

## Step 8: Build Bidirectional LSTM + Self-Attention model

```
Input (30 days × N features)
      │
  BiLSTM(64, return_sequences=True)
      │
  BatchNorm + Dropout(0.25)
      │
  BiLSTM(32, return_sequences=True)
      │
  Self-Attention  ← focuses on the most informative days
      │
  Dropout(0.20)
      │
  Dense(32, relu) → Dense(1)
      │
  Next-day return (scaled)
```

In [ ]:
@tf.keras.utils.register_keras_serializable()
class SelfAttention(Layer):
    """Additive self-attention — learns which timesteps matter most."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attn_weight', shape=(input_shape[-1], 1),
            initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(
            name='attn_bias', shape=(input_shape[1], 1),
            initializer='zeros', trainable=True)
        super().build(input_shape)

    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        return tf.reduce_sum(x * a, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

def build_model(lookback, n_features):
    inp = Input(shape=(lookback, n_features), name='price_input')

    x = Bidirectional(LSTM(64, return_sequences=True), name='bilstm_1')(inp)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)

    x = Bidirectional(LSTM(32, return_sequences=True), name='bilstm_2')(x)
    x = BatchNormalization()(x)
    x = SelfAttention(name='self_attention')(x)
    x = Dropout(0.20)(x)

    x = Dense(32, activation='relu', name='dense_1')(x)
    x = Dropout(0.15)(x)
    out = Dense(1, name='output')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=Adam(learning_rate=0.0005),  # lower LR for stable convergence
        loss='huber',
        metrics=['mae']
    )
    return model

model = build_model(LOOKBACK, n_features)
model.summary()

## Step 9: Train with smart callbacks

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1)
]

print('🚀 Starting training loop...')
print(f'   Samples: {len(X_train)} train | {len(X_test)} test')
print(f'   Features: {n_features} | Lookback: {LOOKBACK} days\n')

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

best_epoch = np.argmin(history.history['val_loss']) + 1
print(f'\n✅ Training complete! Best epoch: {best_epoch} | Best val_loss: {min(history.history["val_loss"]):.6f}')

## Step 10: Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Huber Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('Mean Absolute Error (MAE)')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved training curves plot as 'training_curves.png'")

## Step 11: Evaluation — Directional Accuracy, MAPE

**Directional Accuracy** and **prediction std-dev** are the honest scores here — 50% direction = coin flip, and a std-dev near 0% would mean the model collapsed to predicting a flat/dead line (a common LSTM failure mode worth checking for explicitly).

In [ ]:
print('📊 Testing model accuracy on validation dataset...')
pred_ret_scaled = model.predict(X_test, verbose=0)
pred_return = target_scaler.inverse_transform(pred_ret_scaled).flatten()

# Check raw variance to ensure it's not dead/flat
pred_std = np.std(pred_return * 100)
print(f"📉 Standard deviation of predictions: {pred_std:.4f}%  (should be > 0.05% for active predictions)")

# Directional accuracy
dir_correct = np.sign(pred_return) == np.sign(y_test_ret)
directional_acc = np.mean(dir_correct) * 100

# MAPE on reconstructed prices
pred_price   = prev_close_test * (1.0 + pred_return)
actual_price = prev_close_test * (1.0 + y_test_ret)
mape = np.mean(np.abs((actual_price - pred_price) / actual_price)) * 100

print(f'\n📈 Directional Accuracy: {directional_acc:.2f}%   (50% = random guess)')
print(f'📉 MAPE (Mean Abs Err) : {mape:.4f}%')

## Step 12: Save model + scaler + config

In [ ]:
model.save(MODEL_FILE)
print(f'💾 Saved model weights to: {MODEL_FILE}')

config = {
    'ticker': TICKER,
    'lookback': LOOKBACK,
    'features': FEATURE_COLS,
    'n_features': len(FEATURE_COLS),
    'directional_accuracy': float(round(directional_acc, 2)),
    'mape': float(round(mape, 4))
}
with open(CONFIG_FILE, 'w') as f:
    json.dump(config, f, indent=2)
print(f'💾 Saved feature config metadata to: {CONFIG_FILE}')

print(f'\n📊 Model summary:')
print(f'   Directional Accuracy: {directional_acc:.2f}%')
print(f'   MAPE:                 {mape:.4f}%')
print(f'   Features:             {n_features}')
print(f'   Lookback:              {LOOKBACK} days')

## Step 13: Download files

In [ ]:
try:
    from google.colab import files
    print('📥 Downloading deployment files to local machine...')
    files.download(MODEL_FILE)
    files.download(SCALER_FILE)
    files.download(TARGET_SCALER_FILE)
    files.download(CONFIG_FILE)
    print('🎉 Done! Move all 4 files into your project backend folder.')
except ImportError:
    print('⚠️ Not running in Colab — files saved in the local working directory instead.')

---
## 📌 After training — what to do

1. **Download** all 4 files per asset (run notebook twice — once for Nifty, once for Crude)
2. **Place** all 4 files together in your backend project folder
3. Restart the backend / Streamlit app so it picks up the new model + config
4. The app reads `feature_config_{TICKER}.json` to know exactly which features and lookback this model expects — keep all 4 files from the same run together, never mix-and-match across runs